# 2D Acoustic Wave Simulation with FDTD

This notebook demonstrates a two-dimensional acoustic-wave model inside a rectangular room. It uses the same tested solver as `src/wave_simulation.py` and focuses on the numerical setup, stability condition, wall-arrival times, and visualization.

## Model

We solve

$$
\frac{\partial^2 u}{\partial t^2}
=
c^2\left(\frac{\partial^2 u}{\partial x^2}+\frac{\partial^2 u}{\partial y^2}\right)+S(x,y,t),
$$

with reflective Neumann boundaries

$$
\frac{\partial u}{\partial n}=0.
$$

The explicit FDTD time step is chosen below the two-dimensional CFL stability limit.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from wave_simulation import (
    SimulationConfig,
    run_simulation,
    print_summary,
    save_animation,
    save_snapshot_near_first_wall_hit,
)

## Configuration

In [2]:
config = SimulationConfig(
    room_width_m=6.0,
    room_height_m=4.0,
    temperature_c=20.0,
    target_grid_spacing_m=0.03,
    source_frequency_hz=200.0,
    source_x_m=1.0,
    source_y_m=2.0,
    source_sigma_m=0.06,
    duration_s=0.060,
)
config

SimulationConfig(room_width_m=6.0, room_height_m=4.0, temperature_c=20.0, heat_capacity_ratio=1.4, specific_gas_constant=287.05, target_grid_spacing_m=0.03, cfl_safety=0.9, source_frequency_hz=200.0, source_strength=50000000.0, source_x_m=1.0, source_y_m=2.0, source_sigma_m=0.06, source_ramp_time_s=0.01, duration_s=0.06, snapshot_stride=15)

## Run the simulation

In [3]:
result = run_simulation(config)
print_summary(result)

Sound speed: 343.23 m/s
Grid spacing: dx=0.03000 m, dy=0.03008 m
Time step: 0.00005569 s
Simulation steps: 1079
Theoretical source-to-wall travel times:
    left:   2.91 ms
   right:  14.57 ms
  bottom:   5.83 ms
     top:   5.83 ms


The theoretical wall-arrival times are straight-line source-to-wall distances divided by the speed of sound. With the default source position, the left wall is closest.

## Generate visual outputs

In [4]:
gif_path = save_animation(result, config, ROOT / "assets" / "room_wave.gif")
snapshot_path = save_snapshot_near_first_wall_hit(
    result, config, ROOT / "assets" / "first_wall_arrival.png"
)
print(gif_path)
print(snapshot_path)

/mnt/data/2d-acoustic-wave-simulation/assets/room_wave.gif
/mnt/data/2d-acoustic-wave-simulation/assets/first_wall_arrival.png


## Conclusions

1. The explicit FDTD solver remains stable with a CFL-controlled time step.
2. The wave reaches the nearest (left) wall after approximately 2.91 ms for the default setup.
3. Reflective Neumann boundaries create visible returning wave fronts.
4. The simulation can be reused to explore different source positions, frequencies, room sizes, and grid resolutions.

**Limit:** this is an idealized 2D scalar-wave model with perfectly reflective walls, not a full physical room-acoustics model.